# Pima Indians Diabetes: Naive Bayes and Logistic Regression

This notebook compares a generative classifier, Gaussian Naive Bayes, with a discriminative classifier, Logistic Regression. It uses the same data preparation and fixed stratified train/test split as the accompanying Python file.

**Target:** `Outcome` (0 = no diabetes, 1 = diabetes)

The dataset CSV is included beside this notebook. If it is missing, the first code cell downloads a public copy. The original dataset is available on [Kaggle](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database).

## 1. Imports and dataset

The dataset contains 768 records, eight predictor features, and one target column. This copy of the data has no header row, so the column names are added here.

In [21]:
from pathlib import Path
import urllib.request
import numpy as np
import pandas as pd

DATA_FILE = Path('pima-indians-diabetes.data.csv')
DATA_URL = (
    'https://raw.githubusercontent.com/jbrownlee/Datasets/'
    'master/pima-indians-diabetes.data.csv'
)

if not DATA_FILE.exists():
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)

columns = [
    'Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
    'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'
]
data = pd.read_csv(DATA_FILE, header=None, names=columns)

print(f'Records: {len(data)}')
print(f'Predictor features: {len(columns) - 1}')
print('Outcome counts:')
print(data['Outcome'].value_counts().sort_index())
data.head()

ModuleNotFoundError: No module named 'numpy'

## 2. Split and prepare the data

Use a stratified 80/20 split with seed 42 so the class proportions remain similar in training and testing. In five clinical measurement columns, zero is used as a missing-value code. Replace those zeros using medians computed from the training data only, then standardize features using training-set statistics.

In [ ]:
features = columns[:-1]
X = data[features].to_numpy(dtype=float)
y = data['Outcome'].to_numpy(dtype=int)

rng = np.random.default_rng(42)
train_indices, test_indices = [], []
for label in np.unique(y):
    indices = np.flatnonzero(y == label)
    rng.shuffle(indices)
    test_size = int(round(len(indices) * 0.20))
    test_indices.extend(indices[:test_size])
    train_indices.extend(indices[test_size:])
train_indices = np.array(sorted(train_indices))
test_indices = np.array(sorted(test_indices))

X_clean = X.copy()
zero_means_missing = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for feature in zero_means_missing:
    column = features.index(feature)
    observed_training_values = X_clean[train_indices, column]
    median = np.median(observed_training_values[observed_training_values != 0])
    X_clean[X_clean[:, column] == 0, column] = median

train_mean = X_clean[train_indices].mean(axis=0)
train_scale = X_clean[train_indices].std(axis=0)
train_scale[train_scale == 0] = 1
X_train = (X_clean[train_indices] - train_mean) / train_scale
X_test = (X_clean[test_indices] - train_mean) / train_scale
y_train, y_test = y[train_indices], y[test_indices]

print(f'Train records: {len(y_train)}')
print(f'Test records: {len(y_test)}')
print('Test class counts:', dict(zip(*np.unique(y_test, return_counts=True))))

## 3. Generative classifier: Gaussian Naive Bayes

For each class, Gaussian Naive Bayes estimates a mean and variance for every feature. It combines those likelihoods with the class prior to estimate the probability of each outcome. Its simplifying assumption is that features are conditionally independent given the class.

In [ ]:
def fit_gaussian_naive_bayes(X, y):
    classes = np.unique(y)
    means, variances, priors = [], [], []
    for label in classes:
        class_data = X[y == label]
        means.append(class_data.mean(axis=0))
        variances.append(class_data.var(axis=0) + 1e-9)
        priors.append(len(class_data) / len(y))
    return classes, np.array(means), np.array(variances), np.array(priors)

def predict_gaussian_naive_bayes(model, X):
    classes, means, variances, priors = model
    class_scores = []
    for i, _ in enumerate(classes):
        log_likelihood = -0.5 * np.sum(
            np.log(2 * np.pi * variances[i])
            + ((X - means[i]) ** 2) / variances[i], axis=1
        )
        class_scores.append(log_likelihood + np.log(priors[i]))
    scores = np.array(class_scores).T
    probabilities = np.exp(scores - scores.max(axis=1, keepdims=True))
    probabilities /= probabilities.sum(axis=1, keepdims=True)
    positive_index = list(classes).index(1)
    predictions = classes[np.argmax(probabilities, axis=1)]
    return predictions, probabilities[:, positive_index]

nb_model = fit_gaussian_naive_bayes(X_train, y_train)
nb_predictions, nb_scores = predict_gaussian_naive_bayes(nb_model, X_test)

## 4. Discriminative classifier: Logistic Regression

Logistic Regression directly estimates the probability of the positive class using a sigmoid function. This implementation uses gradient descent and L2 regularization. A probability of 0.5 or higher is classified as Outcome 1.

In [ ]:
def fit_logistic_regression(X, y, learning_rate=0.08, epochs=8000, l2=0.01):
    X_with_intercept = np.column_stack([np.ones(len(X)), X])
    weights = np.zeros(X_with_intercept.shape[1])
    penalty = np.ones_like(weights)
    penalty[0] = 0  # Do not regularize the intercept.

    for _ in range(epochs):
        logits = np.clip(X_with_intercept @ weights, -35, 35)
        probabilities = 1 / (1 + np.exp(-logits))
        gradient = (X_with_intercept.T @ (probabilities - y)) / len(y)
        gradient += l2 * penalty * weights
        weights -= learning_rate * gradient
    return weights

def predict_logistic_regression(weights, X):
    X_with_intercept = np.column_stack([np.ones(len(X)), X])
    scores = 1 / (1 + np.exp(-np.clip(X_with_intercept @ weights, -35, 35)))
    return (scores >= 0.5).astype(int), scores

logistic_weights = fit_logistic_regression(X_train, y_train)
logistic_predictions, logistic_scores = predict_logistic_regression(
    logistic_weights, X_test
)

## 5. Evaluation

Report accuracy, precision, recall, F1-score, ROC-AUC, and confusion counts. The positive class is Outcome 1. The confusion-count order used below is true negatives, false positives, false negatives, true positives.

In [ ]:
def evaluate(y_true, predictions, scores):
    tp = int(np.sum((y_true == 1) & (predictions == 1)))
    tn = int(np.sum((y_true == 0) & (predictions == 0)))
    fp = int(np.sum((y_true == 0) & (predictions == 1)))
    fn = int(np.sum((y_true == 1) & (predictions == 0)))
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    # Compute ROC-AUC from positive-class score ranks; ties get average ranks.
    order = np.argsort(scores)
    ranks = np.empty(len(scores), dtype=float)
    start = 0
    while start < len(order):
        end = start + 1
        while end < len(order) and scores[order[end]] == scores[order[start]]:
            end += 1
        ranks[order[start:end]] = (start + 1 + end) / 2
        start = end
    positives = int(y_true.sum())
    negatives = len(y_true) - positives
    auc = (ranks[y_true == 1].sum() - positives * (positives + 1) / 2) / (positives * negatives)

    return {
        'Accuracy': (tp + tn) / len(y_true),
        'Precision': precision,
        'Recall': recall,
        'F1-score': f1,
        'ROC-AUC': auc,
        'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp,
    }

results = {
    'Gaussian Naive Bayes': evaluate(y_test, nb_predictions, nb_scores),
    'Logistic Regression': evaluate(y_test, logistic_predictions, logistic_scores),
}
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC']
summary = pd.DataFrame({name: {metric: values[metric] for metric in metrics}
                        for name, values in results.items()}).T
summary.round(3)

In [ ]:
for model_name, values in results.items():
    print(
        f"{model_name}: TN={values['TN']}, FP={values['FP']}, "
        f"FN={values['FN']}, TP={values['TP']}"
    )

## 6. Results and discussion

On this fixed split, Logistic Regression had higher accuracy (0.779 vs. 0.747), F1-score (0.630 vs. 0.598), and ROC-AUC (0.837 vs. 0.809). Both models had the same recall (0.537), so each identified the same number of positive test cases. Logistic Regression produced fewer false positives on this split.

The difference is descriptive for this one test split; it should not be treated as proof that one model is always superior. The dataset is small and represents a limited population. These models are for an educational classification exercise, not clinical diagnosis.

## Reproducibility notes

- Split: stratified 80/20; random seed 42.
- Missing values: zeroes in Glucose, BloodPressure, SkinThickness, Insulin, and BMI replaced by training-set medians.
- Scaling: training-set mean and standard deviation.
- Logistic Regression: learning rate 0.08, 8,000 epochs, L2 value 0.01, threshold 0.5.
- Required packages: NumPy and pandas.